# 

Here, examples of ordinal and one-hot encoding are going to be
presented. Importing the libraries.

In [ ]:
import polars as pl
import numpy as np
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

Generating a toy-dataset.

In [ ]:
data = {
    'ID': [1, 2, 3, 4, 5],
    'Color': ['Red', 'Blue', 'Green', 'Blue', 'Red'],
    'Size': ['Small', 'Large', 'Medium', 'Small', 'Medium'],
    'Price': [10, 20, 15, 12, 18]
}

df = pl.DataFrame(data)
print(df)

shape: (5, 4)
┌─────┬───────┬────────┬───────┐
│ ID  ┆ Color ┆ Size   ┆ Price │
│ --- ┆ ---   ┆ ---    ┆ ---   │
│ i64 ┆ str   ┆ str    ┆ i64   │
╞═════╪═══════╪════════╪═══════╡
│ 1   ┆ Red   ┆ Small  ┆ 10    │
│ 2   ┆ Blue  ┆ Large  ┆ 20    │
│ 3   ┆ Green ┆ Medium ┆ 15    │
│ 4   ┆ Blue  ┆ Small  ┆ 12    │
│ 5   ┆ Red   ┆ Medium ┆ 18    │
└─────┴───────┴────────┴───────┘

As the `Color` feature has no order in its values, one-hot encoding is
going to be used. As for the `Size` feature, values assume order, so
ordinal encoding is needed.

In [ ]:
encoder_ohe = OneHotEncoder(sparse_output=False)
ohe_encoded = encoder_ohe.fit_transform(df.select("Color").to_numpy())

ohe_df = pl.DataFrame(
    ohe_encoded,
    schema=[f"color_{c}" for c in encoder_ohe.categories_[0]]
)

order = [["Small", "Medium", "Large"]]
encoder_ord = OrdinalEncoder(categories=order)
exp_encoded = encoder_ord.fit_transform(df.select("Size").to_numpy())

ord_df = pl.concat([df, ohe_df], how="horizontal").with_columns(
    pl.Series("Size_ordinal_encoding", exp_encoded.flatten())
)

print(ord_df)

shape: (5, 8)
┌─────┬───────┬────────┬───────┬────────────┬─────────────┬───────────┬───────────────────────┐
│ ID  ┆ Color ┆ Size   ┆ Price ┆ color_Blue ┆ color_Green ┆ color_Red ┆ Size_ordinal_encoding │
│ --- ┆ ---   ┆ ---    ┆ ---   ┆ ---        ┆ ---         ┆ ---       ┆ ---                   │
│ i64 ┆ str   ┆ str    ┆ i64   ┆ f64        ┆ f64         ┆ f64       ┆ f64                   │
╞═════╪═══════╪════════╪═══════╪════════════╪═════════════╪═══════════╪═══════════════════════╡
│ 1   ┆ Red   ┆ Small  ┆ 10    ┆ 0.0        ┆ 0.0         ┆ 1.0       ┆ 0.0                   │
│ 2   ┆ Blue  ┆ Large  ┆ 20    ┆ 1.0        ┆ 0.0         ┆ 0.0       ┆ 2.0                   │
│ 3   ┆ Green ┆ Medium ┆ 15    ┆ 0.0        ┆ 1.0         ┆ 0.0       ┆ 1.0                   │
│ 4   ┆ Blue  ┆ Small  ┆ 12    ┆ 1.0        ┆ 0.0         ┆ 0.0       ┆ 0.0                   │
│ 5   ┆ Red   ┆ Medium ┆ 18    ┆ 0.0        ┆ 0.0         ┆ 1.0       ┆ 1.0                   │
└─────┴───────┴────────┴──